Algo muito positivo sobre o RI do Itaú é que ele é muito similar ao extrator do banco inter

No Itaú, vai exigir uma estratégia diferente: os títulos mudam levemente, o que dificulta a extração. Tentar mudar o formato para json para ver o que acontece ou usar OCR

observe o padrão, faça casos de correção

Meus desafios são:
1. tenho que considerar que os RIs mudam com o tempo e podem sofrer com alteração no layout
2. Vou ter que ser mais aberta a utilização de OCR. 

In [1]:
import pdfplumber
import pandas as pd
import re
import numpy as np
import json

In [2]:
with pdfplumber.open("Demonstrações Financeiras Itaú - 3T25.pdf") as pdf:
    page = pdf.pages[52]   # página 53 (índice começa em 0)
    text = page.extract_text()

# Remove espaços em branco extras de cada linha ( pré-processamento )
text = "\n".join(line.strip() for line in text.splitlines())

# Define início ( start ) e fim ( end ) do trecho a ser extraído. 
start = text.find("b) Valor Contábil Bruto por Estágios")
end = text.find("Consolidado dos 3 Estágios", start)

trecho = text[start:end]
print(trecho)

# divide em linhas para facilitar a criação do dataframe
linhas = [l.strip() for l in trecho.splitlines() if l.strip()]

b) Valor Contábil Bruto por Estágios
01/01/2025
Saldoem Transferência Transferênciapara Transferência Transferência Aquisição/ Saldoem
Estágio1 31/12/2024 paraEstágio2 Estágio3(1) doEstágio2 doEstágio3 (Liquidação) WriteOff 30/09/2025
PessoasFísicas 347.749 (21.606) (3.019) 34.180 250 36.733 - 394.287
PessoasJurídicas 332.440 (6.181) (1.617) 6.276 494 9.774 - 341.186
UnidadesExternas
196.464 (7.713) (898) 8.245 1.217 (9.816) - 187.499
AméricaLatina
Total 876.653 (35.500) (5.534) 48.701 1.961 36.691 - 922.972
Saldoem Transferência Transferênciapara Transferência Transferência Aquisição/ Saldoem
Estágio2 WriteOff
31/12/2024 paraEstágio1 Estágio3 doEstágio1 doEstágio3 (Liquidação) 30/09/2025
PessoasFísicas 66.468 (34.180) (11.396) 21.606 5.802 (13.814) - 34.486
PessoasJurídicas 13.237 (6.276) (4.470) 6.181 2.018 (1.186) - 9.504
UnidadesExternas
14.004 (8.245) (3.559) 7.713 2.058 (2.241) - 9.730
AméricaLatina
Total 93.709 (48.701) (19.425) 35.500 9.878 (17.241) - 53.720
Saldoem Transferênc

In [44]:
# basicamente, estou sempre criando uma lista vazia nos ifs, e decidindo o que colocar nesta lista vazia ( que seria
# a saída ). Eu posso definir regras do que deve entrar e o que deve sair desta lista

PRODUTOS = [
    "PessoasFísicas",
    "PessoasJurídicas",
]

# verifica se a linha inicia com o nome de um produto
def normalizar(txt: str) -> str:
    return txt.replace(" ", "")

def eh_produto(linha: str) -> bool:
    linha_n = normalizar(linha)
    return any(linha_n.startswith(normalizar(p)) for p in PRODUTOS)

# pega apenas o primeiro e ó último elemento de uma linha
def reduzir_linha_produto(linha: str) -> str:
    partes = linha.split()
    ultimo = partes[-1]

    for produto in PRODUTOS:
        # compara sem espaços
        if normalizar(linha).startswith(normalizar(produto)):
            qtd_palavras_produto = len(produto.split())

            primeiro_valor = partes[qtd_palavras_produto]

            return f"{produto.replace(' ', '')} {primeiro_valor} {ultimo}"

    return linha


# cria um novo dataframe baseado no que foi feito
def reduzir_texto(texto: str) -> str:
    linhas = [l.strip() for l in texto.splitlines() if l.strip()]
    saida = []
    ignorando_bloco = False

    for linha in linhas:


        # Enquanto estiver dentro do bloco, ignore tudo
        if linha.startswith("UnidadesExternas"):
            ignorando_bloco = True
            continue

        # Enquanto estiver dentro do bloco, ignore tudo
        if ignorando_bloco:
            # FIM do bloco
            if linha.startswith("AméricaLatina"):
                ignorando_bloco = False
            continue

        # Regra — remover Total
        if linha.startswith("Total"):
            continue

        # Regra — remover Saldo
        if linha.startswith("Saldo"):
            continue
        
        # Regra — remover Consolidado e o resto
        if linha.startswith("Consolidado"):
            break
        
        if linha[:10].count("/") == 2:
            continue

        
        # para conservar apenas as linhas de estágio
        
        # Estágio 1
        if linha.startswith("Estágio1"):
            saida.append("Estágio1")
            continue

        # Estágio 2
        if linha.startswith("Estágio2"):
            saida.append("Estágio2")
            continue

        # Estágio 3
        if linha.startswith("Estágio3"):
            saida.append("Estágio3")
            continue

        # Redução dos produtos
        if eh_produto(linha):
            saida.append(reduzir_linha_produto(linha))
        else:
            saida.append(linha)

    return "\n".join(saida)



In [45]:
texto_reduzido = reduzir_texto(trecho)
print(texto_reduzido)

b) Valor Contábil Bruto por Estágios
Estágio1
PessoasFísicas 347.749 394.287
PessoasJurídicas 332.440 341.186
Estágio2
PessoasFísicas 66.468 34.486
PessoasJurídicas 13.237 9.504
Estágio3
PessoasFísicas 31.357 26.549
PessoasJurídicas 11.956 10.258


In [23]:
# para criar a coluna de trimestre

from datetime import datetime

padrao_data = re.compile(r"\b\d{2}/\d{2}/\d{4}\b")
datas = padrao_data.findall(texto_reduzido)

print(datas)

def trimestre_from_date(date_str: str) -> str:
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    trimestre_map = {3: "1T", 6: "2T", 9: "3T", 12: "4T"}
    trimestre = trimestre_map.get(dt.month)

    if not trimestre:
        raise ValueError(f"Mês inesperado na data: {date_str}")

    return f"{trimestre}{str(dt.year)[-2:]}"

def extract_anos(texto: str) -> list[str]:
    datas = re.findall(r"\d{2}/\d{2}/\d{4}", texto)

    if len(datas) < 2:
        raise ValueError("Não foi possível encontrar duas datas finais")
    
    datas_finais = datas[-2:]
    return [trimestre_from_date(d) for d in datas_finais]

['01/01/2025', '31/12/2024', '30/09/2025', '31/12/2024', '30/09/2025', '31/12/2024', '30/09/2025']


In [24]:
extract_anos(texto_reduzido)

['4T24', '3T25']

In [ ]:
# o que faz essa parte do código? O que pode ser melhorado?

def parse_linha_estagio(linha):
    partes = linha.split()
    estagio = int(partes[1])

    numeros = partes[2:]

    def conv(x):
        if x == "–":
            return None
        return float(x.replace(".", "").replace(",", "."))

    numeros = [conv(x) for x in numeros]

    return {
        "estagio": estagio,
        "exp_bruta_atual": numeros[0] if len(numeros) > 0 else None,
        "pe_atual": numeros[1] if len(numeros) > 1 else None,
        "exp_bruta_anterior": numeros[2] if len(numeros) > 2 else None,
        "pe_anterior": numeros[3] if len(numeros) > 3 else None,
    }


In [ ]:
# separar em blocos para facilitar a visualização:

# tentar resolver o problema sozinho

padrao = re.compile(
    r"(Estágio1.*?)"
    r"(Estágio2.*?)"
    r"(Estágio3.*)",
    re.S | re.M
)

blocos = {"Estágio1": "", "Estágio2": "", "Estágio3": ""}

for m in padrao.finditer(texto_reduzido):
    for i, estagio in enumerate(["Estágio1", "Estágio2", "Estágio3"], start=1):
        if m.group(i):
            blocos[estagio] = m.group(i).strip()

# resultado
for k, v in blocos.items():
    print(f"\n--- {k} ---\n{v}")


--- Estágio1 ---
Estágio1
PessoasFísicas 347.749 394.287
PessoasJurídicas 332.440 341.186

--- Estágio2 ---
Estágio2
PessoasFísicas 66.468 34.486
PessoasJurídicas 13.237 9.504

--- Estágio3 ---
Estágio3
PessoasFísicas 31.357 26.549
PessoasJurídicas 11.956 10.258
